# Thai AI Paper Feed — Phase B Stage 3: Generate (บน Colab)

notebook นี้ทำ **อย่างเดียว**: ให้แต่ละโมเดลอ่าน paper 60 ใบใน `test.jsonl` แล้วเขียนสรุปออกมา เก็บเป็น JSONL ลง Drive

**ไม่ได้ให้คะแนนที่นี่** — การวัดผล (structure valid / tags / LLM-as-judge) ทำในเครื่อง กับไฟล์ JSONL ที่ได้จากที่นี่

> แยกกันเพราะถ้าอยากเพิ่ม/แก้เกณฑ์วัดผลทีหลัง จะได้ไม่ต้องกลับมาเผา GPU ใหม่

## ผู้เข้าแข่ง

| # | ระบบ | ต้องรันที่นี่ | เวลา |
|---|---|---|---|
| 1 | fine-tuned เคส B (3B, 16-bit) | ✅ | ~10 นาที |
| 2 | fine-tuned เคส A (7B, 4-bit) | ✅ | ~25 นาที |
| 3 | base เคส B (ยังไม่เทรน) | ✅ ตัวควบคุม | ~10 นาที |
| 4 | base เคส A (ยังไม่เทรน) | ✅ ตัวควบคุม | ~25 นาที |
| 5 | Gemini (ครู) | ❌ **มีอยู่แล้ว** ใน `test.jsonl` ฟิลด์ `assistant` | 0 |

**รันทีละเคส แล้ว Runtime → Restart session ก่อนเคสถัดไป** — VRAM ไม่พอโหลดพร้อมกัน

## 0. ติดตั้ง + เตรียม environment

In [ ]:
%%capture
import os
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes xformers triton

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/thai-paper-feed-phase-b'
ADAPTER_DIR = f'{DRIVE_ROOT}/adapters'
EVAL_DIR = f'{DRIVE_ROOT}/eval'
os.makedirs(EVAL_DIR, exist_ok=True)
print('เอาต์พุตจะไปอยู่ที่', EVAL_DIR)

## 1. โหลด test set

In [ ]:
import json

def load_jsonl(path):
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

test_rows = load_jsonl(f'{DRIVE_ROOT}/data/test.jsonl')
print(f'test: {len(test_rows)} rows')
print('ตัวอย่าง id:', [r['id'] for r in test_rows[:3]])

## 2. ฟังก์ชัน generate

จุดที่ตั้งใจออกแบบไว้:

- **`do_sample=False` (greedy)** — ผลซ้ำได้ทุกครั้ง ถ้าสุ่มจะเทียบข้ามโมเดลไม่ได้ว่าใครดีกว่าจริง
- **เก็บ output ดิบทั้งหมด** รวมตัวที่ JSON พัง เพราะ "พังกี่ %" คือหนึ่งในตัวชี้วัด — ไม่ parse ที่นี่
- **จับเวลาต่อใบ** ใช้ประกอบการเลือกผู้ชนะ (ต้นทุน serve ตาม Stage 5.1)
- **resume ได้** ถ้า Colab หลุดกลางทาง รันซ้ำแล้วมันข้ามใบที่ทำไปแล้ว ไม่เริ่มใหม่หมด

In [ ]:
import gc, time, torch
from unsloth import FastLanguageModel

def free_gpu(*objs):
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()

def load_for_inference(model_path, max_seq_length, load_in_4bit):
    """model_path เป็นได้ทั้ง base model id หรือโฟลเดอร์ adapter
    (adapter_config.json บอก base model ไว้อยู่แล้ว Unsloth โหลดต่อให้เอง)"""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=load_in_4bit,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def generate_all(system_name, model_path, out_name, load_in_4bit,
                 max_seq_length=3072, max_new_tokens=1024):
    # max_new_tokens=1024 (เดิม 400): base ยังไม่เทรนเขียนยืดยาว พอชน 400 โดนตัดกลางประโยค
    #   JSON เลยพัง -- เป็นความผิดเรา (cap ต่ำไป) ไม่ใช่โมเดล จึงเพิ่มเป็น 1024 ให้เขียนจบ
    #   (ft ไม่กระทบ: จบเอง ~250-290 token ก่อนชนเพดานอยู่แล้ว + greedy -> ผลเดิมเป๊ะ)
    # max_seq_length=3072: prompt ~1,100 token + new 1024 = ~2,124 ต้องมีที่พอ ไม่งั้น prompt โดนตัด
    out_path = f'{EVAL_DIR}/{out_name}.jsonl'

    # resume: ข้ามใบที่เคยทำแล้ว
    done = set()
    if os.path.exists(out_path):
        done = {r['id'] for r in load_jsonl(out_path)}
        print(f'มีผลเดิมอยู่แล้ว {len(done)} ใบ -> ทำต่อจากของเดิม')

    todo = [r for r in test_rows if r['id'] not in done]
    if not todo:
        print('ครบแล้ว ไม่มีอะไรต้องทำ')
        return out_path

    print(f'=== {system_name} === เหลือ {len(todo)} ใบ')
    model, tokenizer = load_for_inference(model_path, max_seq_length, load_in_4bit)

    with open(out_path, 'a', encoding='utf-8') as f:
        for i, row in enumerate(todo, 1):
            messages = [
                {"role": "system", "content": row["system"]},
                {"role": "user", "content": row["user"]},
            ]
            inputs = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_tensors="pt", return_dict=True,
            ).to(model.device)

            t0 = time.perf_counter()
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,                       # greedy = ผลซ้ำได้
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,   # กัน warning เรื่อง attention mask
            )
            elapsed = time.perf_counter() - t0

            text = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )
            f.write(json.dumps({
                "id": row["id"],
                "system_name": system_name,
                "raw": text,
                "sec": round(elapsed, 2),
            }, ensure_ascii=False) + "\n")
            f.flush()   # เขียนทันทีทุกใบ ถ้า Colab หลุดก็ไม่เสียของที่ทำไปแล้ว

            if i % 10 == 0 or i == len(todo):
                print(f'  {i}/{len(todo)}  ({elapsed:.1f} วิ/ใบ)')

    free_gpu(model, tokenizer)
    print(f'เสร็จ -> {out_path}')
    return out_path

---
## 3. รันทีละเคส

**รันได้ทีละ cell เท่านั้น** แล้ว Runtime → Restart session ก่อนไป cell ถัดไป
(กลับมาต้องรัน cell 0/1/2 ใหม่ก่อน — ตัวแปรหายตอน restart)

เรียงจากเร็วไปช้า: ถ้าโค้ดมีบั๊กจะรู้ตั้งแต่ 10 นาทีแรก ไม่ใช่ 25

### 3.1 fine-tuned เคส B (3B, 16-bit) — เร็วสุด รันตัวนี้ก่อน

In [ ]:
generate_all(
    system_name="ft_case_b",
    model_path=f"{ADAPTER_DIR}/case_b_typhoon2_llama32_3b",
    out_name="gen_ft_case_b",
    load_in_4bit=False,
)

### 3.2 fine-tuned เคส A (7B, 4-bit)

In [ ]:
generate_all(
    system_name="ft_case_a",
    model_path=f"{ADAPTER_DIR}/case_a_typhoon2_qwen25_7b",
    out_name="gen_ft_case_a",
    load_in_4bit=True,
)

### 3.3 base เคส B (ยังไม่เทรน) — ตัวควบคุม

ตอบคำถามว่า **"fine-tune ช่วยจริงไหม"** ซึ่งเป็นคำถามแรกที่คนอ่าน portfolio จะถาม

คาดว่าจะออกมาไม่ดี — อาจไม่เป็น JSON, ตอบเป็นอังกฤษ, หรือพ่นคำอธิบายยาวๆ แทน **นั่นแหละคือผลที่เราต้องการวัด**

In [ ]:
generate_all(
    system_name="base_case_b",
    model_path="typhoon-ai/llama3.2-typhoon2-3b-instruct",
    out_name="gen_base_case_b",
    load_in_4bit=False,
)

### 3.4 base เคส A (ยังไม่เทรน) — ตัวควบคุมฝั่ง 7B

ตัวควบคุมของฝั่ง 7B (พิสูจน์ว่า fine-tune ช่วย เหมือน 3.3 แต่เป็นฝั่ง 7B) กิน GPU ~25 นาที

รันเพื่อให้ได้ตารางครบทั้ง 4 ช่อง (base/ft × A/B)

In [ ]:
# 3.4 base เคส A (7B, 4-bit ดิบ) — ตัวควบคุมฝั่ง 7B
generate_all(
    system_name="base_case_a",
    model_path="scb10x/typhoon2-qwen2.5-7b-instruct",
    out_name="gen_base_case_a",
    load_in_4bit=True,
)

---
## 4. เช็คผลก่อนปิด

ตารางเร็วๆ ว่าแต่ละเคสรันครบไหม + JSON พังกี่ % + เวลา/ใบ
แล้วยก **"ใบที่มีปัญหา" เคสละ 1 ใบ** ให้เห็นว่าโมเดลพลาดตรงไหน

> ยังไม่ใช่การให้คะแนนจริง (นั่นทำในเครื่อง Stage 4) — แค่กันปิด GPU ไปแล้วเพิ่งรู้ว่ามีเคสพังทั้งไฟล์

In [ ]:
import json, re
REQ = {"title_th", "summary_th", "wow_point", "tags"}

def try_parse(raw):
    try:
        return json.loads(raw)
    except Exception:
        pass
    m = re.search(r"\{.*\}", raw, re.S)   # เผื่อโมเดลห่อ JSON ด้วยข้อความอื่น
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return None

def find_problem(rows):
    """หาใบมีปัญหา ไล่จากร้ายแรงสุด -> เบาสุด"""
    for r in rows:                                   # 1) JSON พัง
        if try_parse(r["raw"]) is None:
            return r, "JSON พัง (parse ไม่ได้)"
    for r in rows:                                   # 2) field ไม่ครบ
        o = try_parse(r["raw"])
        if not (isinstance(o, dict) and REQ.issubset(o)):
            return r, "field ไม่ครบ"
    for r in rows:                                   # 3) อาจโดนตัด
        if not r["raw"].rstrip().endswith("}"):
            return r, "อาจโดนตัด/มีข้อความเกิน (ไม่ลงท้ายด้วย })"
    return None, None

# ---------- ตารางสรุป ----------
gen_files = sorted(f for f in os.listdir(EVAL_DIR) if f.startswith("gen_") and f.endswith(".jsonl"))
data = {}
print(f"{'ระบบ':<16}{'ใบ':>4}{'JSON valid':>16}{'field ครบ':>14}{'วิ/ใบ':>9}")
print("-" * 59)
for fn in gen_files:
    name = fn[len("gen_"):-len(".jsonl")]
    rows = load_jsonl(f"{EVAL_DIR}/{fn}")
    data[name] = rows
    ok = sum(try_parse(r["raw"]) is not None for r in rows)
    fok = sum(isinstance(try_parse(r["raw"]), dict) and REQ.issubset(try_parse(r["raw"])) for r in rows)
    avg = sum(r["sec"] for r in rows) / len(rows)
    print(f"{name:<16}{len(rows):>4}{ok:>7}/{len(rows):<2}({100*ok//len(rows):>3}%){fok:>7}/{len(rows):<2}{avg:>9.1f}")

# ---------- ยกตัวอย่างใบมีปัญหา เคสละ 1 ใบ ----------
print("\n" + "=" * 59)
print("ตัวอย่างใบที่มีปัญหา (เคสละ 1 ใบ — ให้เห็นว่าโมเดลพลาดอะไร)")
print("=" * 59)
for name, rows in data.items():
    bad, why = find_problem(rows)
    print(f"\n### {name}")
    if bad is None:
        print("  ไม่เจอปัญหาชัดเจน 👍 (JSON ครบ + field ครบ ทุกใบ)")
    else:
        print(f"  id {bad['id']} — {why}")
        print("  " + bad["raw"][:400].replace("\n", "\n  "))

---
## เสร็จแล้วทำอะไรต่อ

1. ไฟล์ `gen_*.jsonl` อยู่ใน `MyDrive/thai-paper-feed-phase-b/eval/` — โหลดลงเครื่อง
2. **ไม่ต้องแตะ GPU อีกแล้ว** ที่เหลือรันในเครื่องทั้งหมด:
   - วัด automatic metrics (structure valid rate / tags / ศัพท์เทคนิคไม่ถูกแปล)
   - LLM-as-judge ด้วย Claude แบบ blind + ให้คะแนน absolute (ห้ามบอก judge ว่าอันไหนคือเฉลยของ Gemini ไม่งั้น Gemini ชนะโดยอัตโนมัติ)
   - ทำตารางสรุป → เลือกผู้ชนะตามเกณฑ์ Stage 3.3 (คะแนน + ต้นทุน serve)